# conv-channel-sum — ex2: verify per-OC linearity by zeroing one output filter

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `conv-channel-sum`. Running the final beacon cell reports progress against the `CNN: Channel-axis sum semantics` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: Channel-axis sum semantics` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`conv-channel-sum`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "conv-channel-sum"
DD_SUBTOPIC = "CNN: Channel-axis sum semantics"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Conv2d per-OC linearity — quick refresher

Convolution is linear in the kernel. Concretely, output channel `oc` is computed from `weight[oc]` alone — independent of every other output channel slot:

```
y[:, oc, :, :] = sum_ic conv2d_single(x[:, ic, :, :], weight[oc, ic, :, :])
```

**The consequence.** Zeroing out `weight[oc, :, :, :]` for some specific `oc` makes that output channel identically zero everywhere, while every other output channel is bit-exactly unchanged. The 16 filters of a Conv2d(IC, 16, K) are 16 fully independent linear maps stacked along the OC axis — they share input but not weights.

**Why this matters.** It's the mathematical justification for per-channel pruning, filter visualization, and the channel-wise sparsity tricks that ResNet-family lottery-ticket papers exploit. Each `weight[oc]` is its own independent filter; the OC axis is a STACK, not a contraction.

### Exercise 2 — verify per-OC linearity by zeroing one output filter

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze the per-output-channel independence of conv2d by zeroing the kernel slice for one OC slot and verifying that the matching output channel is identically zero while all other OC slots are bit-exactly unchanged.
> Keywords: conv2d, per-channel, linearity, filter-independence
> ```

**KCs targeted:** `conv-per-oc-independence`, `conv-filter-stacking`

Implement `ex2_zero_one_oc(x, weight, target_oc)`. Given input `x: (B, IC, H, W)`, kernel `weight: (OC, IC, KH, KW)`, and an integer `target_oc` in `[0, OC)`, return a **tuple** `(y_full, y_zeroed)` where:

- `y_full = F.conv2d(x, weight)` — the reference output.
- `y_zeroed = F.conv2d(x, w_zeroed)` where `w_zeroed` is a copy of `weight` with **only** `weight[target_oc, :, :, :]` set to zero (every other OC slot unchanged).

**The teaching point.** The OC axis is a STACK of independent filters. Zeroing `weight[target_oc]` must:
1. Set `y_zeroed[:, target_oc, :, :]` to all zeros (no kernel contribution for that filter).
2. Leave EVERY OTHER `y_zeroed[:, other_oc, :, :]` bit-exactly equal to `y_full[:, other_oc, :, :]` — `target_oc`'s weights are independent of every other filter.

**Hint.** Use `weight.clone()` to make a mutable copy, then in-place zero `w_zeroed[target_oc] = 0`. Do NOT modify the original `weight` tensor — the test confirms by re-running `F.conv2d(x, weight)` after your call.

The test then walks every OC slot and asserts the bit-exact match for non-target slots and the all-zero invariant for the target.

In [ ]:
def ex2_zero_one_oc(x: Tensor, weight: Tensor, target_oc: int):
    from torch.nn import functional as F
    y_full = F.conv2d(x, weight)
    w_zeroed = weight.clone()
    w_zeroed[target_oc] = 0.0
    y_zeroed = F.conv2d(x, w_zeroed)
    return y_full, y_zeroed


<details><summary>Solution</summary>

```python
def ex2_zero_one_oc(x: Tensor, weight: Tensor, target_oc: int):
    from torch.nn import functional as F
    y_full = F.conv2d(x, weight)
    w_zeroed = weight.clone()
    w_zeroed[target_oc] = 0.0
    y_zeroed = F.conv2d(x, w_zeroed)
    return y_full, y_zeroed
```

**Why `weight.clone()` and not `.detach()`.** `.clone()` makes a true memory-independent copy; `.detach()` shares storage. Mutating a `.detach()`-ed tensor would mutate the original `weight` — the test catches this via the `weight_snapshot` assertion.

**Why the equality is BIT-EXACT.** Conv2d's output for each OC is `sum_ic conv2d_single(x[:, ic], weight[oc, ic, :, :])`. Different `oc` slots NEVER share weight data, so changing `weight[target_oc]` cannot change any other slot's output — not even by floating-point noise. The test uses `diff == 0.0` exactly, not `allclose`.

**The deeper invariant.** This is the structural justification for filter pruning: you can zero out any subset of OC slots in a trained network and the OTHER channels are bit-identical, so downstream layers see the same data on those channels. (The pruning literature uses this to motivate single-filter ablations as causal: any change in downstream loss is attributable to the pruned filter alone.)

**Contrast with IC.** Zeroing `weight[:, target_ic, :, :, :]` (all OCs at one IC) is a DIFFERENT operation — it kills input channel `target_ic`'s contribution to *every* output channel. OC is a stack; IC is a sum.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()